# Semantic Salah knowledge-graph generator

**What this notebook does.** It reads the raw prayer recordings (one CSV file = one prayer) and writes the **knowledge graph** (the individuals) in the vocabulary of the final **Semantic Salah ontology**. It then checks the result against that ontology.

| | |
|---|---|
| **Inputs** | the folder of raw CSV files · `SemanticSalahOntology.owl` (the final ontology) · `generator_cache.json` (stable Salah/User IDs and prayer-time windows) |
| **Outputs** | `SalahKG_ABox.rdf` (individuals only; declares no import, load it next to the ontology) |
| **Needs** | Python 3, `pandas`, `rdflib` (optional: `requests` for new prayer windows; `owlready2` + Java for the reasoner step) |

**How to run it.** Edit the file names in *Settings*, then *Run all*. Each step prints what it did. Read *Decisions* before you trust the numbers: three data-interpretation choices are switches, not facts.

**Pipeline**
1. Load the ontology (read-only) and check that every term the generator can write exists in it.
2. Read the CSVs and build the individuals (prayer → unit → rakahs → postures).
3. Check the result: structure, and that every term used is declared in the ontology.
4. Preview completeness using the ontology's own definitions (no reasoner needed).
5. Optional: run a real OWL reasoner on a few prayers.

**Why the notebook opens the ontology, but never saves it.** The old notebook loaded an old ontology, redefined classes inside the code and saved a combined file; that is how names drifted (`Postures`, `SalahUnitCategory`, `Fajar`). Here the ontology is the single source of truth: it is read-only, it is used to *check* what the generator writes (step 1 and 3) and to *evaluate* completeness (step 4–5), and the output contains only individuals.


In [ ]:
import os, re, json, datetime
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD
import rdflib
rdflib.NORMALIZE_LITERALS = False   # keep date-times exactly as written (see write_graph)

ONT = Namespace("http://www.semanticsalah.com/ontology/")
ONTOLOGY_IRI = URIRef("http://www.semanticsalah.com")          # the final ontology
KG_IRI = URIRef("http://www.semanticsalah.com/kg")             # this individuals-only file
# False (default): the graph file declares no import, so Protege does not try to download the ontology; load the graph
# next to SemanticSalahOntology.owl. True: add owl:imports of the ontology IRI (use once the ontology is published there).
ADD_ONTOLOGY_IMPORT = False
SCHEMA_PERSON = URIRef("http://schema.org/Person")

# --- mapping tables (CSV label -> ontology term) -------------------------------------------
PRESCRIBED = {"Fajar": 2, "Maghrib": 3, "Zuhr": 4, "Asar": 4, "Isha": 4}      # farz rakahs per prayer
PRAYER = {"Fajar": "Fajr", "Maghrib": "Maghrib", "Zuhr": "Zuhr", "Asar": "Asar", "Isha": "Isha"}
POSTURE_TYPE = {"Qayam": "QayamType", "Ruku": "RukuType", "Qoum": "QoumType",
                "Sajda1": "SajdaType", "Sajda2": "SajdaType",
                "Jalsa1": "JalsaType", "Jalsa2": "JalsaType", "Tashahud": "TashahudType"}
# Recorded events that are NOT one of the six postures. They stay out of the posture chain
# (Takbir = the opening takbir at the start of the prayer, confirmed by the data owner; its timestamp is the prayer's
# start time. RY = read as Raf' al-Yadayn, the hand raising, a brief gesture rather than a posture.)
NOT_POSTURES = {"Takbir", "RY"}
# Jalsa2 = the short sitting before standing up for the next rakah (after rakah 1, and after rakah 3 of a 4-rakah
# prayer); confirmed by the data-collection team. The ontology allows it as an optional ending of those rakahs, so
# the default keeps it in the posture chain. True would drop it from the data (only needed for comparisons).
SKIP_JALSA2 = False
# Rakah boundaries. The recorded Curr_rakah counter agrees with the posture sequence in 39 of 42 prayers and is wrong
# in S7, S37 and S40 (e.g. it reads 3,4,3 or jumps early; confirmed as an error by the data-collection team).
# True: rakahs are derived from the posture sequence (a new rakah starts at a Qayam that follows Sajda2, Jalsa2 or
# Tashahud; a Qayam recorded twice in a row, e.g. Qayam-RY-Qayam, is one standing). False: use Curr_rakah as recorded.
RAKAH_FROM_SEQUENCE = True
RAKAH_ENDERS = {"Sajda2", "Jalsa2", "Tashahud"}


## Settings
All file names in one place.

In [ ]:
DATA_FOLDER   = "FYP2_DATA-COLLECTION"        # folder with one sub-folder per participant, each with CSV files
ONTOLOGY_FILE = "../ontology/SemanticSalahOntology.owl"    # the final ontology (read-only)
CACHE_FILE    = "generator_cache.json"         # IDs and prayer windows of the existing graph
OUTPUT_FILE   = "../kg/SalahKG_ABox.rdf"        # the generated knowledge graph (individuals only)

# optional reasoner step (slow; needs Java and owlready2). Use a few prayers, e.g. ["S14", "S3"]
RUN_REASONER      = False
REASONER_PRAYERS  = ["S14"]
EXTRA_ONTOLOGIES  = []                         # e.g. ["SemanticStructuredActivity.owl"] to include its disjointness axiom

# optional: also run the diagnosis rules (SWRL module) and print what was missed / added. Needs the reasoner step.
SHOW_DIAGNOSIS = False
RULES_FILE     = "../ontology/SemanticSalahRules.owl"


## Decisions you can change (the switches in the first code cell)
These are interpretation choices about the recordings. Please confirm each against your protocol.

- **`NOT_POSTURES`** – `Takbir` and `RY` are recorded events but not one of the six postures, so they are kept out of the posture chain. `Takbir` (42 rows, once at the start of every prayer, about 1 second) is the opening takbir: **confirmed**; its timestamp is the prayer's start time, so nothing is lost. `RY` (37 rows, always between Qayam and Ruku, between Ruku and Qoum, or on standing after a Tashahud, under a second) is read as Raf' al-Yadayn, the hand raising: a gesture, not a posture *(please confirm the wording)*.
- **`SKIP_JALSA2`** – `Jalsa2` (26 rows) is the short sitting before standing up for the next rakah, after rakah 1 and after rakah 3 of a 4-rakah prayer (**confirmed by the data-collection team**; it matches the data: it always follows `Sajda2` and precedes `Qayam`). The ontology allows it as an optional ending of those rakahs, so the default (`False`) keeps it in the data as a `Jalsa` posture. `True` would drop it, which is only useful for comparisons.
- **`RAKAH_FROM_SEQUENCE`** – the recorded `Curr_rakah` counter is wrong in three prayers (S7, S37, S40; confirmed as an error by the data-collection team), so rakahs are derived from the posture sequence: a new rakah starts at a `Qayam` that follows `Sajda2`, `Jalsa2` or `Tashahud`, and a `Qayam` recorded twice in a row (S7, S37: Qayam, RY, Qayam) is one standing. In the other 39 prayers this gives exactly the recorded numbers. Result: S7 and S37 are 3-rakah recordings, S40 is an ordinary 4-rakah prayer (no extra rakah). `False` uses `Curr_rakah` as recorded.
- **`PRESCRIBED`** – prescribed rakahs come from the prayer type (Fajr 2, Maghrib 3, Zuhr/Asr/Isha 4), not from the `Entered_Rakah` column (the two differ in 4 files, possibly travellers' prayers).

## Helper functions
Parsing of timestamps, reading the CSV files, and the optional prayer-window lookup.

In [ ]:
LATITUDE, LONGITUDE = 33.738045, 73.084488       # Islamabad
WINDOW_KEYS = {"Isha": ("Isha", "Fajr"), "Maghrib": ("Maghrib", "Isha"), "Asar": ("Asr", "Maghrib"),
               "Zuhr": ("Dhuhr", "Asr"), "Fajar": ("Fajr", "Sunrise")}

def fetch_window(first_ts, label):
    """Prescribed time window from the aladhan.com API (same logic as the original notebook).
    Returns (start, end) strings, or None if the API cannot be reached."""
    try:
        import requests
        date, _ = str(first_ts).strip().split("_")
        y, m, d = date.split("-")
        r = requests.get("https://api.aladhan.com/v1/calendar/%s/%s?latitude=%s&longitude=%s" % (y, int(m), LATITUDE, LONGITUDE), timeout=20)
        if r.status_code != 200: return None
        timings = r.json()["data"][int(d) - 1]["timings"]
        a, b = WINDOW_KEYS[label]
        return timings[a], timings[b]
    except Exception:
        return None

def write_graph(g, path):
    """Write the graph as RDF/XML. HermiT (the reasoner in Protege) cannot read xsd:dateTime values with more than
    3 fractional-second digits, so any longer fraction is cut to milliseconds."""
    text = g.serialize(format="pretty-xml")
    text = re.sub(r"(T\d\d:\d\d:\d\d\.\d{3})\d+(<)", r"\1\2", text)
    open(path, "w", encoding="utf-8").write(text)

def parse_ts(s):
    """'2022-02-28_14:01:32:581' (or with a space) -> datetime"""
    s = str(s).strip().replace("_", " ")
    date, clock = s.split(" ")
    h, m, sec, ms = clock.split(":")
    y, mo, d = date.split("-")
    return datetime.datetime(int(y), int(mo), int(d), int(h), int(m), int(sec), int(ms) * 1000)

def dt_literal(dt):
    return Literal(dt.strftime("%Y-%m-%dT%H:%M:%S.") + "%03d" % (dt.microsecond // 1000), datatype=XSD.dateTime)

def load_csvs(folder):
    """one CSV = one prayer; returns list of (first_timestamp, dataframe) in a stable order"""
    out = []
    for d in sorted(x for x in os.listdir(folder) if os.path.isdir(os.path.join(folder, x))):
        for f in sorted(x for x in os.listdir(os.path.join(folder, d)) if x.lower().endswith(".csv")):
            df = pd.read_csv(os.path.join(folder, d, f))
            if df.shape[1] != 6:
                continue
            out.append((str(df["Timestamp"].iloc[0]), df))
    return out

def segment_rakahs(df, sid, log):
    """Return [(timestamp, activity, rakah_number)] for the postures of one prayer (recorded events are left out)."""
    rows = []; k = 0; prev = None; changed = False
    for _, r in df.iterrows():
        a = str(r["Activity"])
        if a in NOT_POSTURES or (SKIP_JALSA2 and a == "Jalsa2"):
            log["events_skipped"][a] = log["events_skipped"].get(a, 0) + 1; continue
        recorded = int(r["Curr_rakah"])
        if RAKAH_FROM_SEQUENCE:
            if a == "Qayam":
                if prev is None or prev in RAKAH_ENDERS: k += 1                    # standing up to start a new rakah
                elif prev == "Qayam":                                               # the same standing recorded twice
                    log.setdefault("duplicate_qayam_dropped", []).append(sid); changed = True; continue
            k = max(k, 1)
            if k != recorded: changed = True
        else:
            k = recorded
        rows.append((parse_ts(r["Timestamp"]), a, k)); prev = a
    if changed: log.setdefault("rakah_labels_corrected", []).append(sid)
    return rows


## The ontology (read-only)
`load_ontology` opens the ontology. `check_mappings` verifies that every class, property and individual the generator can write exists in it. `check_vocabulary` does the same for the generated graph afterwards.

In [ ]:
USED_CLASSES = ["Salah", "SalahUnit", "Rakah", "Posture", "SalahTimeWindow"] + sorted(set(PRAYER.values()))
USED_PROPERTIES = ["performs", "isPerformedBy", "hasSalahType", "isStartedAtTime", "isCompletedAtTime", "hasPrescribedWindow",
                   "hasWindowStartTime", "hasWindowEndTime", "hasSalahUnit", "hasSalahUnitType", "hasUnitRakahCount",
                   "hasRecordedRakahCount", "containsRakah", "hasRakahNumber", "hasStartingPosture", "hasPostureCount",
                   "containsPosture", "rakahFollowedByRakah", "postureStartsAtTime", "postureEndsAtTime", "followedBy",
                   "hasPostureType"]
USED_INDIVIDUALS = ["Farz"] + sorted(set(POSTURE_TYPE.values())) + sorted(set(PRAYER.values()))

def load_ontology(path):
    """Load the final ontology READ-ONLY (RDF/XML)."""
    onto = Graph()
    onto.parse(path, format="xml")
    return onto

def check_mappings(onto):
    """Every ontology term the generator can write must be declared in the ontology.
    Returns a list of missing terms (empty = OK). This is what catches name drift such as Fajar/Fajr."""
    missing = []
    for n in USED_CLASSES:
        if (ONT[n], RDF.type, OWL.Class) not in onto: missing.append("class " + n)
    for n in USED_PROPERTIES:
        if not any((ONT[n], RDF.type, t) in onto for t in (OWL.ObjectProperty, OWL.DatatypeProperty)): missing.append("property " + n)
    for n in USED_INDIVIDUALS:
        if (ONT[n], RDF.type, OWL.NamedIndividual) not in onto: missing.append("individual " + n)
    return missing

def check_vocabulary(g, onto):
    """Every ontology-namespace term actually used in the generated graph must be declared in the ontology
    (or declared inside the generated file itself, like the four location terms)."""
    declared_here = {s for t in (OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty) for s in g.subjects(RDF.type, t)}
    used = set()
    for s, p, o in g:
        if str(p).startswith(str(ONT)): used.add(p)
        if p == RDF.type and str(o).startswith(str(ONT)): used.add(o)
        if p in (ONT.hasSalahType, ONT.hasPostureType, ONT.hasSalahUnitType): used.add(o)
    missing = []
    for t in sorted(used, key=str):
        if t in declared_here: continue
        if not any((t, RDF.type, k) in onto for k in (OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty, OWL.NamedIndividual)):
            missing.append(str(t).replace(str(ONT), ""))
    return missing


## Building the individuals
`build_abox` creates, for each prayer: the Salah, its type (and class, e.g. `Fajr`), the unit (`Farz`), the rakahs and the postures, with all counts the completeness rules need. Rules it follows:
- one CSV file = one prayer = one Farz unit;
- the prescribed number of rakahs (`hasUnitRakahCount`) comes from the prayer type; the recorded number (`hasRecordedRakahCount`) and the posture counts are counted from the data;
- postures are linked in order with `followedBy`; rakahs with `rakahFollowedByRakah` (never to themselves);
- times are written as `xsd:dateTime`; person identifiers are replaced by `User<n>`.

In [ ]:
def build_abox(prayers, cache):
    g = Graph(); g.bind("ontology", ONT); g.bind("owl", OWL); g.bind("rdfs", RDFS)
    g.add((KG_IRI, RDF.type, OWL.Ontology))
    if ADD_ONTOLOGY_IMPORT: g.add((KG_IRI, OWL.imports, ONTOLOGY_IRI))
    else: g.add((KG_IRI, RDFS.comment, Literal("Individuals of the Semantic Salah ontology (http://www.semanticsalah.com). It declares no import: load it together with SemanticSalahOntology.owl.")))
    # location terms used by the data (not part of the core ontology yet, declared here)
    for c in ("SalahLocation",): g.add((ONT[c], RDF.type, OWL.Class))
    for p in ("performedAtLocation", "windowDeterminedBy"): g.add((ONT[p], RDF.type, OWL.ObjectProperty))
    g.add((ONT.hasLocationName, RDF.type, OWL.DatatypeProperty))
    g.add((ONT.performedAtLocation, RDFS.domain, ONT.Salah)); g.add((ONT.performedAtLocation, RDFS.range, ONT.SalahLocation))
    g.add((ONT.windowDeterminedBy, RDFS.domain, ONT.SalahTimeWindow)); g.add((ONT.windowDeterminedBy, RDFS.range, ONT.SalahLocation))
    g.add((ONT.hasLocationName, RDFS.domain, ONT.SalahLocation)); g.add((ONT.hasLocationName, RDFS.range, XSD.string))
    used_ids = {v["salah"] for v in cache.values()}
    next_n = 1 + max([int(i[1:]) for i in used_ids] + [2])
    persons = {}
    log = {"prayers": 0, "postures": 0, "events_skipped": {}, "rakahs": 0}
    def ind(iri, *types):
        g.add((iri, RDF.type, OWL.NamedIndividual))
        for t in types: g.add((iri, RDF.type, t))
        return iri
    for first_ts, df in prayers:
        c = cache.get(first_ts)
        if c: sid, user = c["salah"], c["user"]
        else:
            sid = "S%d" % next_n; next_n += 1
            pid = str(df["person_id"].iloc[0]); persons.setdefault(pid, "User%d" % (len(persons) + 100))
            user = persons[pid]; c = {}
        label = str(df["Namaz"].iloc[0]); pname = PRAYER[label]; prescribed = PRESCRIBED[label]
        salah = ind(ONT[sid], ONT.Salah, ONT[pname]); g.add((salah, ONT.hasSalahType, ONT[pname]))
        person = ind(ONT[user], SCHEMA_PERSON); g.add((person, ONT.performs, salah)); g.add((salah, ONT.isPerformedBy, person))
        g.add((salah, ONT.isStartedAtTime, dt_literal(parse_ts(df["Timestamp"].iloc[0]))))
        g.add((salah, ONT.isCompletedAtTime, dt_literal(parse_ts(df["Timestamp"].iloc[-1]))))
        loc = ind(ONT[c.get("location", "Islamabad_Pakistan")], ONT.SalahLocation); g.add((loc, ONT.hasLocationName, Literal(c.get("location", "Islamabad_Pakistan"))))
        g.add((salah, ONT.performedAtLocation, loc))
        tw = ind(ONT[sid + "_TimeWindow"], ONT.SalahTimeWindow); g.add((salah, ONT.hasPrescribedWindow, tw)); g.add((tw, ONT.windowDeterminedBy, loc))
        w = (c["window_start"], c["window_end"]) if c.get("window_start") else fetch_window(first_ts, label)
        if w: g.add((tw, ONT.hasWindowStartTime, Literal(w[0]))); g.add((tw, ONT.hasWindowEndTime, Literal(w[1])))
        else: log.setdefault("no_window", []).append(sid)
        unit = ind(ONT[sid + "_FZ_Salah"], ONT.SalahUnit); g.add((salah, ONT.hasSalahUnit, unit))
        g.add((unit, ONT.hasSalahUnitType, ONT.Farz))
        g.add((unit, ONT.hasUnitRakahCount, Literal(prescribed, datatype=XSD.integer)))     # prescribed by the prayer type
        # ---- postures: skip recorded events that are not postures --------------------------
        rows = segment_rakahs(df, sid, log)
        rakahs = {}; order = []
        for _, a, k in rows:
            if k not in rakahs: rakahs[k] = []; order.append(k)
        prev_p = None; prev_k = None; prev_rakah = None; counters = {}
        for i, (ts, a, k) in enumerate(rows):
            counters[k] = counters.get(k, 0) + 1
            p = ind(ONT["%s_FZ_Rakah%d_P%d" % (sid, k, counters[k])], ONT.Posture)
            g.add((p, ONT.postureStartsAtTime, dt_literal(ts)))
            g.add((p, ONT.hasPostureType, ONT[POSTURE_TYPE[a]]))
            if prev_p is not None:
                g.add((prev_p, ONT.followedBy, p)); g.add((prev_p, ONT.postureEndsAtTime, dt_literal(ts)))
            rakahs[k].append(p); prev_p = p
        for idx, k in enumerate(order):
            rk = ind(ONT["%s_FZ_Salah_Rakah%d" % (sid, k)], ONT.Rakah); g.add((unit, ONT.containsRakah, rk))
            g.add((rk, ONT.hasRakahNumber, Literal(k, datatype=XSD.integer)))
            g.add((rk, ONT.hasStartingPosture, rakahs[k][0]))
            g.add((rk, ONT.hasPostureCount, Literal(len(rakahs[k]), datatype=XSD.integer)))
            for p in rakahs[k]: g.add((rk, ONT.containsPosture, p))
            if idx > 0: g.add((prev_rakah, ONT.rakahFollowedByRakah, rk))          # only between different rakahs
            prev_rakah = rk
        g.add((unit, ONT.hasRecordedRakahCount, Literal(len(order), datatype=XSD.integer)))
        log["prayers"] += 1; log["postures"] += len(rows); log["rakahs"] += len(order)
    return g, log


## Structural checks
No reasoner needed: self-loops, untyped postures, wrong counts, postures that belong to no rakah.

In [ ]:
def validate(g):
    """cheap structural checks (no reasoner needed)"""
    problems = []
    for s, o in g.subject_objects(ONT.rakahFollowedByRakah):
        if s == o: problems.append("self-loop on rakahFollowedByRakah: %s" % s)
    for p in g.subjects(RDF.type, ONT.Posture):
        if g.value(p, ONT.hasPostureType) is None: problems.append("untyped posture: %s" % p)
    for rk in g.subjects(RDF.type, ONT.Rakah):
        n = len(list(g.objects(rk, ONT.containsPosture)))
        if int(g.value(rk, ONT.hasPostureCount)) != n: problems.append("posture count mismatch: %s" % rk)
    contained = {p for rk in g.subjects(RDF.type, ONT.Rakah) for p in g.objects(rk, ONT.containsPosture)}
    for p in g.subjects(RDF.type, ONT.Posture):
        if p not in contained: problems.append("posture not in any rakah: %s" % p)
    return problems


## Run: load, build, check, write

In [ ]:
onto = load_ontology(ONTOLOGY_FILE)
missing = check_mappings(onto)
assert not missing, "terms the generator writes are missing from the ontology: %s" % missing
print("ontology loaded, all mapped terms exist in it")

cache   = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}
prayers = load_csvs(DATA_FOLDER)
g, log  = build_abox(prayers, cache)

print("prayers:", log["prayers"], "| rakahs:", log["rakahs"], "| postures:", log["postures"])
print("recorded events kept out of the posture chain:", log["events_skipped"])
print("rakah numbers re-derived from the posture sequence in:", sorted(set(log.get("rakah_labels_corrected", []))))
print("duplicate Qayam dropped in:", sorted(set(log.get("duplicate_qayam_dropped", []))))
print("prayers without a prescribed window:", log.get("no_window", []))
problems = validate(g)
print("structural problems:", len(problems)); [print("  ", p) for p in problems[:10]]
unknown = check_vocabulary(g, onto)
print("terms used but not declared in the ontology:", unknown)

write_graph(g, OUTPUT_FILE)
long_fractions = len(re.findall(r">\d{4}-\d\d-\d\dT[\d:]+\.\d{4,}<", open(OUTPUT_FILE, encoding="utf-8").read()))
print("written:", OUTPUT_FILE, "(", len(g), "triples )", "| dateTime literals with more than 3 fractional digits:", long_fractions)

## Completeness preview (no reasoner)
This reads the ontology's own `Completed…Rakah` / `Incompleted…Rakah` definitions and evaluates them on the generated rakahs, plus the unit rule (fewer rakahs recorded than prescribed). **The OWL reasoner is authoritative**; this is a fast, readable sanity check that gives the same answer on the cases we compared.

In [ ]:
import re as _re
from collections import Counter as _Counter
from rdflib.collection import Collection as _Collection
from rdflib import BNode as _BNode

def completeness_preview(g, onto):
    m = Graph(); m += onto; m += g
    ln = lambda x: str(x).split("/")[-1].split("#")[-1]
    def conj(x):
        if isinstance(x, _BNode) and (x, OWL.intersectionOf, None) in m:
            out = []
            for q in _Collection(m, m.value(x, OWL.intersectionOf)): out += conj(q)
            return out
        return [x]
    def ptype(p): return ln(m.value(p, ONT.hasPostureType)).replace("Type", "Posture")
    def chain_ok(p, x):          # does posture p satisfy "Name and followedBy some (...)"
        for q in conj(x):
            if isinstance(q, URIRef):
                if ptype(p) != ln(q): return False
            elif m.value(q, OWL.onProperty) == ONT.followedBy:
                if not any(chain_ok(n, m.value(q, OWL.someValuesFrom)) for n in m.objects(p, ONT.followedBy)): return False
        return True
    size = {2: "TwoSalahUnit", 3: "ThreeSalahUnit", 4: "FourSalahUnit"}
    unit_of = {r: u for u in m.subjects(RDF.type, ONT.SalahUnit) for r in m.objects(u, ONT.containsRakah)}
    defs = []
    for c in m.subjects(RDF.type, OWL.Class):
        if not _re.match(r"(Incompleted|Completed)(Two|Three|Four)UnitRakah\d$", ln(c)): continue
        for e in m.objects(c, OWL.equivalentClass):
            alts = list(_Collection(m, m.value(e, OWL.unionOf))) if (e, OWL.unionOf, None) in m else [e]
            for d in alts: defs.append((ln(c).startswith("Completed"), conj(d)))
    verdict = {}
    for r in g.subjects(RDF.type, ONT.Rakah):
        u = unit_of.get(r)
        if u is None: continue
        unit_class = size.get(int(m.value(u, ONT.hasUnitRakahCount))); hit = set()
        for is_complete, req in defs:
            ok = True
            for q in req:
                p = m.value(q, OWL.onProperty)
                if p == ONT.belongsToUnit and ln(m.value(q, OWL.someValuesFrom)) != unit_class: ok = False
                elif p == ONT.hasRakahNumber and int(m.value(q, OWL.hasValue)) != int(m.value(r, ONT.hasRakahNumber)): ok = False
                elif p == ONT.hasPostureCount and int(m.value(q, OWL.hasValue)) != int(m.value(r, ONT.hasPostureCount)): ok = False
                elif p == ONT.containsPosture and not any(chain_ok(n, m.value(q, OWL.someValuesFrom)) for n in m.objects(r, ONT.containsPosture)): ok = False
            if ok: hit.add("complete" if is_complete else "incomplete")
        verdict[r] = "both (contradiction)" if len(hit) == 2 else (next(iter(hit)) if hit else "undecided")
    prayers = []
    for s in g.subjects(RDF.type, ONT.Salah):
        for u in g.objects(s, ONT.hasSalahUnit):
            pres, rec = int(g.value(u, ONT.hasUnitRakahCount)), int(g.value(u, ONT.hasRecordedRakahCount))
            v = _Counter(verdict[r] for r in g.objects(u, ONT.containsRakah))
            prayers.append((ln(s), pres, rec, dict(v)))
    return verdict, prayers

def print_completeness(verdict, prayers):
    print("rakahs:", dict(_Counter(verdict.values())))
    ok = [p for p in prayers if p[2] == p[1] and set(p[3]) <= {"complete"}]
    print("prayers:", len(prayers), "| all rakahs complete and as many rakahs as prescribed:", len(ok))
    print("units with FEWER rakahs than prescribed (IncompleteSalahUnit):", [(p[0], "%d of %d" % (p[2], p[1])) for p in prayers if p[2] < p[1]])
    print("units with MORE rakahs than prescribed (no rule yet):", [(p[0], "%d of %d" % (p[2], p[1])) for p in prayers if p[2] > p[1]])
    print("prayers with a rakah the ontology cannot classify:", [p[0] for p in prayers if p[3].get("undecided")])


In [ ]:
verdict, prayer_rows = completeness_preview(g, onto)
print_completeness(verdict, prayer_rows)

## Optional: real OWL reasoner
HermiT through `owlready2` (needs Java). It checks consistency and infers the classes (e.g. `TwoSalahUnit`, `IncompleteSalahUnit`, `CompletedFourUnitRakah2`). It is slow, so give it only a few prayers. Set `RUN_REASONER = True` in *Settings* to use it. With `SHOW_DIAGNOSIS = True` it also loads the optional rules module (`SemanticSalahRules.owl`) and prints *what* was missed (a posture, or a whole rakah) or added (an extra rakah).

In [ ]:
def run_reasoner(ontology_file, g, prayer_ids, extra_ontologies=(), show_diagnosis=False):
    import tempfile
    from owlready2 import get_ontology, sync_reasoner_hermit, default_world, OwlReadyInconsistentOntologyError
    keep = Graph()
    for s, p, o in g:
        name = str(s).replace(str(ONT), "")
        if any(name == i or name.startswith(i + "_") for i in prayer_ids): keep.add((s, p, o))
    merged = Graph(); merged.parse(ontology_file, format="xml")
    for f in extra_ontologies: merged.parse(f, format="xml")
    merged += keep
    merged.remove((None, OWL.imports, None))          # everything is merged here; nothing to download
    text = merged.serialize(format="xml")
    text = _re.sub(r"(T\d\d:\d\d:\d\d\.\d{3})\d+<", r"\1<", text)      # HermiT cannot read >3 fractional-second digits
    path = tempfile.NamedTemporaryFile(suffix=".owl", delete=False).name
    open(path, "w", encoding="utf-8").write(text)
    onto = get_ontology("file://" + path).load()
    try:
        with onto: sync_reasoner_hermit(infer_property_values=show_diagnosis)
    except OwlReadyInconsistentOntologyError:
        print("RESULT: the ontology + data are INCONSISTENT"); return
    print("RESULT: consistent. Unsatisfiable classes:", list(default_world.inconsistent_classes()))
    nm = lambda c: getattr(c, "name", str(c))
    for i in prayer_ids:
        s = default_world[str(ONT) + i]
        print(i, "->", sorted(nm(c) for c in s.is_a))
        for u in default_world[str(ONT) + "SalahUnit"].instances():
            if u.name.startswith(i + "_"): print("   unit  ->", sorted(nm(c) for c in u.is_a))
        for r in sorted((x for x in default_world[str(ONT) + "Rakah"].instances() if x.name.startswith(i + "_")), key=lambda x: x.name):
            print("   %-24s ->" % r.name.replace(i + "_FZ_Salah_", ""), sorted(nm(c) for c in r.is_a if nm(c) != "Rakah"))
            if show_diagnosis:
                miss = [nm(v) for v in default_world[str(ONT) + "hasMissedPosture"][r]]
                if miss: print("   %-24s    diagnosis: missed posture" % "", sorted(miss))
        if show_diagnosis:
            for u in default_world[str(ONT) + "SalahUnit"].instances():
                if u.name.startswith(i + "_"):
                    d = [nm(v) for v in default_world[str(ONT) + "hasMissedRakah"][u]] + [nm(v) for v in default_world[str(ONT) + "hasExtraRakah"][u]]
                    if d: print("   unit diagnosis (missed or extra rakah):", sorted(d))


In [ ]:
if RUN_REASONER:
    run_reasoner(ONTOLOGY_FILE, g, REASONER_PRAYERS, EXTRA_ONTOLOGIES + ([RULES_FILE] if SHOW_DIAGNOSIS else []), SHOW_DIAGNOSIS)
else:
    print("reasoner step skipped (set RUN_REASONER = True in Settings)")

## Reading the results
- **rakah level** – `complete` / `incomplete` come from the posture sequence and the posture count; `undecided` means the ontology has no definition that fits (for example the recorded `Jalsa2`).
- **unit level** – a unit with fewer recorded rakahs than the prayer type prescribes is an `IncompleteSalahUnit`.
- **prayer level** – obligation comes from the prayer class (e.g. `Fajr` is an `ObligatorySalah`); the ontology does not yet infer a completion status for a whole prayer.
- The graph also uses four location terms that are not part of the core ontology (`SalahLocation`, `performedAtLocation`, `windowDeterminedBy`, `hasLocationName`); they are declared inside the generated file.